# Computing an Exact Interaction

This notebook showcases how for a simple 3 player game a single interaction is computed.

In [1]:
from dataclasses import dataclass
from typing import Any, ClassVar

import jax
import jax.numpy as jnp
from jaxtyping import Float, Shaped, UInt8
from ndlike.protected_axis.jax import JaxAxisProtected

from shapiq.game import Game, PackedFinitePlayerGame
from shapiq.players import FinitePlayers
from shapiq.typing import typechecked

In [2]:
@typechecked
@dataclass
class TableGame(PackedFinitePlayerGame[Any, Float[jax.Array, "..."]], JaxAxisProtected[jax.Array]):
    values: Float[jax.Array, "*batch 2**{_players.ids.shape[-1]}"]
    _players: Shaped[FinitePlayers, "#*batch"]
    protected_axes: ClassVar[dict[str, int]] = {"values": 1}

    @classmethod
    def _pytree_child_names(cls) -> tuple[str, ...]:
        # Players are dynamic, but not an equally batched protected field.
        return ("values", "_players")

    @property
    @typechecked
    def players(self: Shaped[Game, "*batch"]) -> Shaped[FinitePlayers, "#*batch"]:
        return self._players

    @typechecked
    def _eval_packed(
        self: Shaped[Game, "#*batch"],
        coalition: UInt8[jax.Array, "#*batch ({self.players.n_players}+7)//8"],
    ) -> Float[jax.Array, "*batch"]:
        n_players = self.players.n_players
        index_dtype = jnp.uint64 if jax.config.x64_enabled else jnp.uint32
        if n_players > jnp.iinfo(index_dtype).bits:
            msg = "Table indices exceed the enabled integer width."
            raise ValueError(msg)
        shifts = 8 * jnp.arange(coalition.shape[-1], dtype=index_dtype)
        index = jnp.sum(coalition.astype(index_dtype) << shifts, axis=-1, dtype=index_dtype)
        index &= jnp.array((1 << n_players) - 1, dtype=index_dtype)
        shape = jnp.broadcast_shapes(self.shape, index.shape)
        table = jnp.broadcast_to(self.values, (*shape, self.values.shape[-1]))
        index = jnp.broadcast_to(index, shape)
        return jnp.take_along_axis(table, index[..., None], axis=-1)[..., 0]

In [3]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


Now we take a weighting scheme and put it into the discrete derivative computation

In [4]:
coalitions.shape

(8, 3)

In [5]:
# Shapley Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDShapleyWeight

ddcomputer = WeightedDiscreteDerivative(
    weights=DDShapleyWeight(), game=g, interaction=jnp.array([1, 0, 0], dtype=bool), order=1
)
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:, None, :])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(), ddshapley.shape

(Array([2.3333302], dtype=float32), Array(2.3333302, dtype=float32), (8, 1))

In [6]:
# Shapley Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDShapleyWeight

ddcomputer = WeightedDiscreteDerivative(
    weights=DDShapleyWeight(),
    game=g,
    interaction=jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=bool),
    order=1,
)
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:, None, :])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(), ddshapley.shape

(Array([2.3333302, 3.3333287, 4.3333273], dtype=float32),
 Array(9.999987, dtype=float32),
 (8, 3))

In [7]:
# Shapley Interaction Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDShapleyWeight

ddcomputer = WeightedDiscreteDerivative(
    weights=DDShapleyWeight(),
    game=g,
    interaction=jnp.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=bool),
    order=2,
)
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:, None, :])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(), ddshapley.shape

(Array([1.4999989, 1.4999989, 1.4999989], dtype=float32),
 Array(4.4999967, dtype=float32),
 (8, 3))

In [8]:
# AlphaBetaShapley Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDAlphaBetaWeight

alpha_beta_shapley_weight = DDAlphaBetaWeight(
    alpha=jnp.array([1, 2, 5])[:, None], beta=jnp.array([1, 2, 5])[:, None]
)
ddcomputer = WeightedDiscreteDerivative(
    weights=alpha_beta_shapley_weight,
    game=g,
    interaction=jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=bool),
    order=1,
)
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:, None, None, :])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(axis=-1), ddshapley.shape

(Array([[2.3333302, 3.3333287, 4.3333273],
        [2.3000016, 3.3000023, 4.300003 ],
        [2.2727294, 3.2727304, 4.2727313]], dtype=float32),
 Array([9.999987, 9.900007, 9.818192], dtype=float32),
 (8, 3, 3))

In [9]:
# AlphaBetaShapley Interaction Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDAlphaBetaWeight

alpha_beta_shapley_weight = DDAlphaBetaWeight(
    alpha=jnp.array([1, 2, 5])[:, None], beta=jnp.array([1, 2, 5])[:, None]
)
ddcomputer = WeightedDiscreteDerivative(
    weights=alpha_beta_shapley_weight,
    game=g,
    interaction=jnp.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=bool),
    order=2,
)
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:, None, None, :])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(axis=-1), ddshapley.shape

(Array([[1.4999989, 1.4999989, 1.4999989],
        [1.5000002, 1.5000002, 1.5000002],
        [1.4999993, 1.4999993, 1.4999993]], dtype=float32),
 Array([4.4999967, 4.500001 , 4.499998 ], dtype=float32),
 (8, 3, 3))

In [10]:
# Banzhaf Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDBanzhafWeight

banzhaf_weight = DDBanzhafWeight()
ddcomputer = WeightedDiscreteDerivative(
    weights=banzhaf_weight,
    game=g,
    interaction=jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=bool),
    order=1,
)
ddbanzhaf = ddcomputer(coalition=coalitions[:, None, :])
banzhaf_values = jnp.sum(ddbanzhaf, axis=0)
banzhaf_values, banzhaf_values.sum(), ddbanzhaf.shape

(Array([2.25, 3.25, 4.25], dtype=float32), Array(9.75, dtype=float32), (8, 3))

In [11]:
# Banzhaf Interaction Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDBanzhafWeight

banzhaf_weight = DDBanzhafWeight()
ddcomputer = WeightedDiscreteDerivative(
    weights=banzhaf_weight,
    game=g,
    interaction=jnp.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=bool),
    order=2,
)
ddbanzhaf = ddcomputer(coalition=coalitions[:, None, :])
banzhaf_values = jnp.sum(ddbanzhaf, axis=0)
banzhaf_values, banzhaf_values.sum(), ddbanzhaf.shape

(Array([1.5, 1.5, 1.5], dtype=float32), Array(4.5, dtype=float32), (8, 3))

In [12]:
# Weighted Banzhaf Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDWeightedBanzhafWeight

weighted_banzhaf_weight = DDWeightedBanzhafWeight(weight=jnp.array([0.5, 0.3, 0.2])[None])
ddcomputer = WeightedDiscreteDerivative(
    weights=weighted_banzhaf_weight,
    game=g,
    interaction=jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=bool),
    order=1,
)
weighed_banzhaf = ddcomputer(coalition=coalitions[:, None, :])
weighed_banzhaf_values = jnp.sum(weighed_banzhaf, axis=0)
weighted_banzhaf_shapes = weighed_banzhaf.shape
print("Weighted Banzhaf Values:\n", weighed_banzhaf)
print("Weighted Banzhaf Totals:\n", weighed_banzhaf_values)
print("Weighted Banzhaf Shapes:\n", weighted_banzhaf_shapes)

Weighted Banzhaf Values:
 [[0.25       0.97999996 1.9200001 ]
 [0.         0.63       0.64000005]
 [0.5        0.         0.64000005]
 [0.         0.         0.24000001]
 [0.5        0.63       0.        ]
 [0.         0.45000002 0.        ]
 [1.         0.         0.        ]
 [0.         0.         0.        ]]
Weighted Banzhaf Totals:
 [2.25      2.6899998 3.4400003]
Weighted Banzhaf Shapes:
 (8, 3)


In [13]:
# Weighted Banzhaf Interaction Implementation
from shapiq.game.discretederivative import WeightedDiscreteDerivative
from shapiq.game.weights import DDWeightedBanzhafWeight

weighted_banzhaf_weight = DDWeightedBanzhafWeight(weight=jnp.array([0.5, 0.3, 0.2])[None])
ddcomputer = WeightedDiscreteDerivative(
    weights=weighted_banzhaf_weight,
    game=g,
    interaction=jnp.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=bool),
    order=2,
)
weighed_banzhaf = ddcomputer(coalition=coalitions[:, None, :])
weighed_banzhaf_values = jnp.sum(weighed_banzhaf, axis=0)
weighted_banzhaf_shapes = weighed_banzhaf.shape
print("Weighted Banzhaf Values:\n", weighed_banzhaf)
print("Weighted Banzhaf Totals:\n", weighed_banzhaf_values)
print("Weighted Banzhaf Shapes:\n", weighted_banzhaf_shapes)

Weighted Banzhaf Values:
 [[0.5 0.7 0.8]
 [0.  0.  0.4]
 [0.  0.6 0. ]
 [0.  0.  0. ]
 [1.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]]
Weighted Banzhaf Totals:
 [1.5 1.3 1.2]
Weighted Banzhaf Shapes:
 (8, 3)
